# 3-bob. Mashinali o'qitish xavfsizlik nuqtai nazaridan
## Kengaytirilgan amaliy ish: faol va tasodifiy belgilashni taqqoslash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/03-bob/3-3_faol-va-tasodifiy-belgilashni-taqqoslash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 3-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur 5000 ta belgilanmagan tarmoq ulanishi (shundan taxminan 3 foizi hujum) "hovuzi"dan iborat vaziyatni modellashtiradi. Tahlilchi dastlab 20 ta yozuvni belgilagan, keyin esa har safar yana 20 tadan, jami 300 tagacha yozuvni belgilaydi. Ikki strategiya taqqoslanadi: tasodifiy tanlash va noaniqlik bo'yicha tanlash. Model sifati alohida sinov to'plamida aniqlik-to'liqlik egri chizig'i ostidagi yuza (PR-AUC) bilan baholanadi; bu ko'rsatkich nomutanosib ma'lumotlar uchun mos ekanini 5-bobda ko'ramiz. Natija tasodifiylikka bog'liq bo'lmasligi uchun har bir tajriba besh marta takrorlanadi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

In [ ]:
# 1. Sun'iy ma'lumotlar: 8000 ta ulanish, shundan 3 foizi hujum
X, y = make_classification(n_samples=8000, n_features=20, n_informative=8, weights=[0.97],
                           class_sep=1.5, flip_y=0.01, random_state=3)
X_hovuz, y_hovuz, X_sinov, y_sinov = X[:5000], y[:5000], X[5000:], y[5000:]

def tajriba(strategiya, boshlang=20, qadam=20, byudjet=300, urug=0):
    rng = np.random.default_rng(urug)
    # boshlang'ich to'plam: tahlilchi 20 ta tasodifiy yozuvni belgilagan (kamida 2 ta hujum bilan)
    hujum = rng.choice(np.where(y_hovuz == 1)[0], 2, replace=False)
    oddiy = rng.choice(np.where(y_hovuz == 0)[0], boshlang - 2, replace=False)
    belgili = list(hujum) + list(oddiy)
    natija = {}
    while True:
        model = RandomForestClassifier(100, random_state=0, n_jobs=-1).fit(X_hovuz[belgili], y_hovuz[belgili])
        natija[len(belgili)] = average_precision_score(y_sinov, model.predict_proba(X_sinov)[:, 1])
        if len(belgili) >= byudjet:
            return natija
        qolgan = np.setdiff1d(np.arange(len(X_hovuz)), belgili)
        if strategiya == "tasodifiy":
            yangi = rng.choice(qolgan, qadam, replace=False)
        else:   # noaniqlik: model eng ikkilanayotgan (ehtimoli 0,5 ga yaqin) yozuvlar
            p = model.predict_proba(X_hovuz[qolgan])[:, 1]
            yangi = qolgan[np.argsort(np.abs(p - 0.5))[:qadam]]
        belgili += list(yangi)

In [3]:
# 2. Har bir strategiya uchun 5 marta takrorlab, o'rtachasini olamiz
for nom in ("tasodifiy", "noaniqlik"):
    natijalar = [tajriba(nom, urug=u) for u in range(5)]
    qatorlar = [f"{n}: {np.mean([r[n] for r in natijalar]):.2f}" for n in (20, 60, 100, 200, 300)]
    print(f"{nom:<10} PR-AUC  " + "   ".join(qatorlar))

tasodifiy  PR-AUC  20: 0.11   60: 0.16   100: 0.29   200: 0.43   300: 0.47


noaniqlik  PR-AUC  20: 0.11   60: 0.38   100: 0.62   200: 0.75   300: 0.79


In [4]:
# 3. Barcha 5000 yozuv belgilansa (eng yaxshi holat)
toliq = RandomForestClassifier(100, random_state=0, n_jobs=-1).fit(X_hovuz, y_hovuz)
print(f"barcha 5000 ta belgi bilan PR-AUC: {average_precision_score(y_sinov, toliq.predict_proba(X_sinov)[:, 1]):.2f}")

barcha 5000 ta belgi bilan PR-AUC: 0.77


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
tasodifiy  PR-AUC  20: 0.11   60: 0.16   100: 0.29   200: 0.43   300: 0.47
noaniqlik  PR-AUC  20: 0.11   60: 0.38   100: 0.62   200: 0.75   300: 0.79
barcha 5000 ta belgi bilan PR-AUC: 0.77
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 3-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.